# LandShift D3.5 — Section 4.3
# Ecosystem Multifunctionality

| | |
|---|---|
| **Project** | LandShift — Horizon Europe, Grant Agreement No. 101182007 |
| **Task** | T3.3 — Mapping ecosystem services linked with past, current, and projected LULUCF |
| **Lead** | University of Geneva (UNIGE) |
| **Collaborators** | POLIMI, UNIBAS, ECoE, BFRS |
| **Reference** | D3.5 Ecosystem services & landscape ecology, §4.3 |
| **FAIR compliance** | Outputs are open GeoTIFF / CSV / PNG / MD; notebook is fully reproducible |

---

## Purpose

This notebook implements the **Tier 3 LandShift product** described in §4.3 of D3.5:
a spatially explicit **Ecosystem Service (ES) Multifunctionality Index** that describes
the simultaneous provision of multiple ecosystem services across the study area.

It assumes that the **InVEST ecosystem service rasters** have already been computed
(Tier-2 output of `ecosystem_services/ecosystem_services.ipynb` in the
[WP3 folder of the HE-LandShift repository](https://github.com/LivingEarthLab/HE-LandShift/tree/main/WP3))
and are available on disk.  The service keys are the same as in the §4.1 notebook.

The notebook implements the three complementary multifunctionality metrics from §4.3.3,
following the established ecological multifunctionality literature
(Byrnes et al. 2014, doi:10.1038/s41559-017-0461-7;
Zavaleta et al. 2010, doi:10.1126/science.1215442;
Manning et al. 2018, doi:10.1016/j.jenvman.2020.111068):

| Metric | Symbol | Description | D3.5 ref |
|---|---|---|---|
| Averaged Multifunctionality Index | MF_avg | Mean normalised performance across all services | §4.3.3.1 |
| Threshold-Based Multifunctionality Index | MF_thr | Count of services exceeding a performance threshold T | §4.3.3.2 |
| Weighted Multifunctionality Index | MF_w | Stakeholder-priority-weighted sum of normalised services | §4.3.3.3 |

### Workflow (D3.5 §4.3)

| Step | Description | D3.5 Reference |
|------|-------------|----------------|
| 1 | Load and harmonise InVEST service rasters (common grid, common valid-pixel mask) | §4.3.1 |
| 2 | Directional harmonisation (higher = better) | §4.3.2.1 |
| 3 | Min–max normalisation to [0, 1] | §4.3.2.2 |
| 4 | Compute MF_avg, MF_thr, MF_w | §4.3.3 |
| 5 | Sensitivity and uncertainty analysis | §4.3.4 |
| 6 | Hotspots, trade-offs, zonal summaries (ecosystem types, optional administrative zones) | §4.3.5 |
| 7 | Export deliverables: **GeoTIFF rasters**, tables, report | §4.3.6 |

> ⚠️ **All indices are computed on a common set of pixels** — pixels where every service has a value (and that
> lie inside the MAES ecosystem extent, if provided).  Otherwise MF_avg, MF_thr and MF_w would be built from
> different sets of services in different places and could not be compared.
> Min–max normalisation is *relative to the study area*: the indices describe relative performance
> (D3.5 §4.3.5), not absolute service levels.

---

## How to use this notebook

1. Run **Section 0** to install and import dependencies.
2. Edit **Section 1** (Configuration) — set the `SERVICES` dictionary (path, **direction**, weight), thresholds, optional MAES raster and zones.
3. Run all cells top-to-bottom. Outputs land in `OUTPUT_DIR`.
4. Keep `DEMO_MODE = True` to test with synthetic data (Section A);
   set `DEMO_MODE = False` for production.
5. To compare **other years or scenarios** on the same scale, set `NORM_BOUNDS_CSV` to the
   `table_normalisation_bounds_*.csv` written by the reference run (Step 3).

> ℹ️  **Relationship to §4.1:** The InVEST service rasters are the same Tier-2 outputs consumed by the §4.1
> notebook, and the MAES extent layer (if provided) defines the common grid.

---
## Section 0 — Dependencies

In [ ]:
# Install required packages if not already present.
# Comment out this cell if working in a pre-configured environment.
import subprocess, sys

PACKAGES = ["numpy", "pandas", "rasterio", "matplotlib", "scipy", "affine", "geopandas"]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("All packages ready.")

In [ ]:
import datetime
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.ticker import PercentFormatter, ScalarFormatter

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# display() is a Jupyter built-in; provide a plain-print fallback so cells
# also work when the notebook is executed as a plain Python script.
try:
    from IPython.display import display
except ImportError:
    def display(obj):  # noqa: F811
        print(obj)

print(f"numpy    {np.__version__}")
print(f"pandas   {pd.__version__}")
print(f"rasterio {rasterio.__version__}")
print("Imports OK.")

---
## Section 1 — Configuration

**Edit the variables in this cell before running the rest of the notebook.**

In [ ]:
# ── InVEST service rasters (Tier-2 outputs of ecosystem_services.ipynb) ───────
# One entry per service.  Fields:
#   path      : raster path, or a LIST of paths (e.g. one per pollinator guild) combined pixel-wise by `combine`
#   label     : human-readable name (figures, tables)
#   unit      : unit of the raster values (figure labels)
#   direction : True  → higher value = better service performance
#               False → higher value = worse performance (e.g. nutrient EXPORT); inverted after
#                       normalisation (D3.5 §4.3.2.1)
#   weight    : stakeholder priority for MF_w (§4.3.3.3); normalised to sum 1 inside the notebook;
#               0 = not used in MF_w
#   kind      : 'stock' | 'flow' | 'pressure' | 'index'   (a 'pressure' must have direction=False)
#   group     : services of the same group overlap (e.g. carbon storage vs sequestration) → double counting
#   combine   : 'mean' | 'sum' when `path` is a list (default 'mean')
#
# ⚠️ Layers must be DENSITIES (per hectare) or dimensionless indices so that pixels are comparable.
SERVICES = {
    "carbon_storage": dict(
        path=Path("inputs/invest/carbon/c_storage_bas.tif"),
        label="Carbon storage", unit="Mg C ha⁻¹", direction=True, weight=2.0,
        kind="stock", group="carbon"),
    "sediment_avoided_export": dict(
        path=Path("inputs/invest/sediment/avoided_export.tif"),
        label="Sediment retention (avoided export)", unit="t ha⁻¹ yr⁻¹", direction=True, weight=1.0,
        kind="flow", group="sediment"),
    "n_total_export": dict(
        path=Path("inputs/invest/ndr/n_total_export.tif"),
        label="Nitrogen export", unit="kg ha⁻¹ yr⁻¹", direction=False, weight=1.0,
        kind="pressure", group="nutrient"),
    "habitat_quality": dict(
        path=Path("inputs/invest/habitat/quality_c.tif"),
        label="Habitat quality", unit="index (0–1)", direction=True, weight=2.0,
        kind="index", group="habitat"),
    "pollinator_supply": dict(
        path=sorted(Path("inputs/invest/pollination").glob("pollinator_supply_*.tif")),
        label="Pollinator supply (mean of guilds)", unit="index (0–1)", direction=True, weight=1.5,
        kind="index", group="pollination", combine="mean"),
}

# ── Optional MAES raster: defines the common grid and the ecosystem extent, enables per-type summaries ──
# Set to None to skip; the grid is then taken from the first service raster.
MAES_RASTER        = Path("PATH_TO_MAES_RASTER")   # or None
MAES_LOOKUP_CSV    = Path("PATH_TO/maes_lookup.csv")        # or None
MAES_EXCLUDE_CODES = [0, 255]

# ── Optional administrative / basin zones for zonal summaries (§4.3.1: administrative regions,
#    river basins, NUTS units).  Any vector format readable by geopandas; None to skip.
ZONES_VECTOR     = None                # e.g. Path("inputs/nuts3.gpkg")
ZONES_NAME_FIELD = None                # attribute used as zone name (None = running number)

# ── Output
OUTPUT_DIR      = Path("outputs/section_4_3") #TBC
ACCOUNTING_YEAR = 2025 #TBC
LIVING_SPACE    = "Basilicata" #TBC

# ── Spatial settings
ALIGN_SERVICES         = True                 # re-grid service rasters onto the common grid when they differ
RESAMPLE_METHOD        = Resampling.bilinear
SERVICE_NODATA         = None                 # None = read from raster metadata
PIXEL_AREA_HA_OVERRIDE = None

# ── Normalisation (§4.3.2.2)
# NORM_METHOD: 'minmax'   → (x − min) / (max − min)   [default, D3.5 equation]
#              'quantile' → rank-based; robust to extreme outliers (used for the sensitivity analysis)
NORM_METHOD = "minmax"
# To compare years / scenarios on the SAME scale, point to the bounds table of the reference run
# (table_normalisation_bounds_<year>.csv).  Values outside the reference range are clipped to [0, 1].
NORM_BOUNDS_CSV = None

# ── Threshold-based index (§4.3.3.2) and its sensitivity range (§4.3.4; D3.5 suggests 40 / 50 / 60 %)
THRESHOLD_PRIMARY = 0.50
THRESHOLD_RANGE   = [0.30, 0.40, 0.50, 0.60, 0.70]

# ── Hotspots (§4.3.5): quantiles of MF_avg defining cold spots and hotspots (relative classes)
HOTSPOT_QUANTILES = (25, 75)

# ── Quality-check thresholds
CORR_FLAG          = 0.80   # |r| above which two services are flagged as redundant (§4.3.4)
MIN_RETAINED_PCT   = 90.0   # minimum share of the mapped area with values for ALL services
ROBUST_R_MIN       = 0.80   # minimum Pearson r in the normalisation / weight sensitivity checks

# ── Demo mode
DEMO_MODE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Living Space       : {LIVING_SPACE}")
print(f"  Accounting year    : {ACCOUNTING_YEAR}")
print(f"  Services           : {list(SERVICES)}")
print(f"  Threshold (primary): {THRESHOLD_PRIMARY}")
print(f"  Output folder      : {OUTPUT_DIR.resolve()}")
print(f"  Demo mode          : {DEMO_MODE}")

---
## Section A — Synthetic Demo Data *(skip in production)*

When `DEMO_MODE = True` this section generates small synthetic GeoTIFFs that mimic real MAES and InVEST outputs
(real MAES codes 101–107, one nutrient-export layer that must be inverted, a list of two pollinator guild rasters, a few
missing pixels, and — if `geopandas` is installed — four demo zones), so the full notebook can be executed without any
real input files.  Set `DEMO_MODE = False` in Section 1 to run on real data.

In [ ]:
if DEMO_MODE:
    from rasterio.transform import from_origin
    from scipy.ndimage import uniform_filter

    DEMO_DIR = OUTPUT_DIR / "_demo_inputs"
    DEMO_DIR.mkdir(parents=True, exist_ok=True)

    ROWS, COLS = 200, 200
    DEMO_RES   = 10.0
    DEMO_TF    = from_origin(2600000, 1200000, DEMO_RES, DEMO_RES)
    DEMO_CRS   = "EPSG:3035"
    rng        = np.random.default_rng(42)

    def write_demo_tif(path, arr, dtype, nodata):
        with rasterio.open(path, "w", driver="GTiff", height=ROWS, width=COLS, count=1, dtype=dtype,
                           crs=DEMO_CRS, transform=DEMO_TF, nodata=nodata) as dst:
            dst.write(arr.astype(dtype), 1)
        return path

    # ── MAES raster with the real LandShift codes ────────────────────────────
    maes_labels_demo = {101: "Urban (Settlements and other artificial areas)", 102: "Cropland",
                        103: "Grassland", 104: "Forest and woodlands", 107: "Wetlands"}
    maes_colors_demo = {101: "#d9a38f", 102: "#ffd21f", 103: "#d3e6b0", 104: "#7cb518", 107: "#b2e3e3"}
    codes_demo = np.array(list(maes_labels_demo), dtype=np.uint8)
    band_h = ROWS // len(codes_demo)
    maes_arr_demo = np.zeros((ROWS, COLS), dtype=np.uint8)
    for i, code in enumerate(codes_demo):
        maes_arr_demo[i * band_h:(i + 1) * band_h, :] = code
    noise = rng.random((ROWS, COLS)) < 0.08
    maes_arr_demo[noise] = rng.choice(codes_demo, size=noise.sum())
    demo_maes_path = write_demo_tif(DEMO_DIR / "maes_level2_demo.tif", maes_arr_demo, np.uint8, 0)

    demo_lookup_path = DEMO_DIR / "maes_lookup_demo.csv"
    pd.DataFrame({"maes_code": list(maes_labels_demo), "ecosystem_type": list(maes_labels_demo.values()),
                  "maes_color": [maes_colors_demo[c] for c in maes_labels_demo]}).to_csv(demo_lookup_path, index=False)

    # ── InVEST-like service rasters: per-type means + noise, smoothed ────────
    svc_means = {
        "carbon_storage"         : {101: 5,    102: 60,   103: 80,   104: 180,  107: 120},
        "sediment_avoided_export": {101: 0.2,  102: 1.2,  103: 2.5,  104: 4.0,  107: 0.8},
        "n_total_export"         : {101: 8.0,  102: 25,   103: 6.0,  104: 2.0,  107: 3.0},
        "habitat_quality"        : {101: 0.05, 102: 0.30, 103: 0.60, 104: 0.85, 107: 0.75},
        "pollinator_guild_A"     : {101: 0.05, 102: 0.40, 103: 0.60, 104: 0.55, 107: 0.30},
        "pollinator_guild_B"     : {101: 0.04, 102: 0.30, 103: 0.50, 104: 0.65, 107: 0.35},
    }
    demo_invest_dir = DEMO_DIR / "invest"; demo_invest_dir.mkdir(exist_ok=True)
    demo_paths = {}
    for name, means in svc_means.items():
        arr = np.zeros((ROWS, COLS), dtype=np.float32)
        for code, mu in means.items():
            m = maes_arr_demo == code
            arr[m] = rng.normal(mu, mu * 0.15, m.sum()).clip(0)
        arr = uniform_filter(arr, size=5).astype(np.float32)
        if name in ("habitat_quality", "pollinator_guild_A", "pollinator_guild_B"):
            arr = arr.clip(0, 1)
        if name == "sediment_avoided_export":                      # a few missing pixels
            arr[rng.random((ROWS, COLS)) < 0.03] = -9999.0
        demo_paths[name] = write_demo_tif(demo_invest_dir / f"{name}.tif", arr, np.float32, -9999.0)

    # ── Four demo zones (quadrants) as a GeoPackage, if geopandas is available ──
    try:
        import geopandas as gpd
        from shapely.geometry import box
        x0, y0 = 2600000, 1200000 - ROWS * DEMO_RES
        xm, ym = x0 + COLS * DEMO_RES / 2, y0 + ROWS * DEMO_RES / 2
        xe, ye = x0 + COLS * DEMO_RES, y0 + ROWS * DEMO_RES
        zones = gpd.GeoDataFrame({"zone_name": ["North-West", "North-East", "South-West", "South-East"]},
                                 geometry=[box(x0, ym, xm, ye), box(xm, ym, xe, ye), box(x0, y0, xm, ym), box(xm, y0, xe, ym)],
                                 crs=DEMO_CRS)
        demo_zones_path = DEMO_DIR / "zones_demo.gpkg"
        zones.to_file(demo_zones_path, driver="GPKG")
        ZONES_VECTOR, ZONES_NAME_FIELD = demo_zones_path, "zone_name"
    except ImportError:
        print("  (geopandas not installed — demo zones skipped)")

    # ── Override configuration ───────────────────────────────────────────────
    MAES_RASTER, MAES_LOOKUP_CSV = demo_maes_path, demo_lookup_path
    for key in ("carbon_storage", "sediment_avoided_export", "n_total_export", "habitat_quality"):
        SERVICES[key]["path"] = demo_paths[key]
    SERVICES["pollinator_supply"]["path"] = [demo_paths["pollinator_guild_A"], demo_paths["pollinator_guild_B"]]
    ALIGN_SERVICES = False

    print("Demo rasters created:")
    print(f"  MAES  : {MAES_RASTER.name}")
    for k, v in demo_paths.items():
        print(f"  {k}: {v.name}")
else:
    print("DEMO_MODE = False — using paths defined in Section 1.")

---
## Section 2 — Helper Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1  Pixel area
# ─────────────────────────────────────────────────────────────────────────────
def get_pixel_area_ha(raster_path: Path) -> float:
    """Return pixel area in hectares from the raster affine transform (metre units assumed)."""
    if PIXEL_AREA_HA_OVERRIDE is not None:
        return float(PIXEL_AREA_HA_OVERRIDE)
    with rasterio.open(raster_path) as src:
        return (abs(src.transform.a) * abs(src.transform.e)) / 10_000.0


# ─────────────────────────────────────────────────────────────────────────────
# 2.2  Spatial alignment  (§4.3.1)
# ─────────────────────────────────────────────────────────────────────────────
def align_to_reference(src_path, ref_path, dst_path, resampling=Resampling.bilinear,
                       src_nodata=None, dst_nodata=-9999.0):
    """
    Reproject and resample src_path to exactly match the CRS, resolution, extent and pixel
    alignment of ref_path.  Writes the result (float32) to dst_path.

    Implements §4.3.1: 'all raster layers are harmonised to ensure common CRS, identical spatial
    resolution, identical spatial extent and consistent pixel alignment.'
    """
    with rasterio.open(ref_path) as ref:
        ref_crs_, ref_tf_, ref_h_, ref_w_ = ref.crs, ref.transform, ref.height, ref.width
    with rasterio.open(src_path) as src:
        raw    = src.read(1).astype(np.float32)
        src_nd = src.nodata if src_nodata is None else src_nodata
        bad = np.isnan(raw)
        if src_nd is not None:
            bad |= np.isclose(raw, src_nd)
        raw = np.where(bad, dst_nodata, raw)
        dst_data = np.full((ref_h_, ref_w_), dst_nodata, dtype=np.float32)
        reproject(source=raw, destination=dst_data,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_tf_,       dst_crs=ref_crs_,
                  resampling=resampling, src_nodata=dst_nodata, dst_nodata=dst_nodata)
    with rasterio.open(dst_path, "w", driver="GTiff", height=ref_h_, width=ref_w_, count=1,
                       dtype=np.float32, crs=ref_crs_, transform=ref_tf_, nodata=dst_nodata) as dst:
        dst.write(dst_data, 1)
    return Path(dst_path)


def load_aligned_layer(path, tag):
    """Read one raster on the common grid (float32, NaN = missing); re-grid it first if needed."""
    path = Path(path)
    with rasterio.open(path) as src:
        res_m, crs, shape, tf = abs(src.transform.a), str(src.crs), (src.height, src.width), src.transform
        nd_src = src.nodata if SERVICE_NODATA is None else SERVICE_NODATA
    same_grid = (crs == str(ref_crs) and shape == ref_shape
                 and np.allclose(tuple(tf)[:6], tuple(ref_tf)[:6], atol=1e-6))
    info = {"file": path.name, "res_m": res_m, "crs": crs, "shape": shape, "same_grid": same_grid, "aligned": False}
    load_path, nd_load = path, nd_src
    if not same_grid:
        if not ALIGN_SERVICES:
            raise ValueError(f"{path.name}: grid differs from the reference grid ({res_m:.1f} m, {shape}, {crs}) "
                             "and ALIGN_SERVICES is False.")
        load_path = align_to_reference(path, REF_RASTER, ALIGNED_DIR / f"{tag}_aligned.tif",
                                       RESAMPLE_METHOD, src_nodata=nd_src)
        nd_load, info["aligned"] = -9999.0, True
    with rasterio.open(load_path) as src:
        arr = src.read(1).astype(np.float32)
    bad = np.isnan(arr)
    if nd_load is not None:
        bad |= np.isclose(arr, nd_load)
    return np.where(bad, np.nan, arr).astype(np.float32), info


# ─────────────────────────────────────────────────────────────────────────────
# 2.3  Normalisation  (§4.3.2.2)
# ─────────────────────────────────────────────────────────────────────────────
def normalise(arr: np.ndarray, method: str = "minmax", bounds=None) -> np.ndarray:
    """
    Normalise a 2-D float array to [0, 1].  NaN pixels are preserved.

    method = 'minmax'  : ES' = (ES − min) / (max − min)   [D3.5 §4.3.2.2]
                         `bounds=(lo, hi)` fixes the scale (e.g. that of a reference year); values
                         outside the bounds are clipped to [0, 1].
    method = 'quantile': rank-based normalisation robust to outliers (no external bounds)
    """
    valid = arr[~np.isnan(arr)]
    if valid.size == 0:
        return arr
    if method == "minmax":
        lo, hi = (float(valid.min()), float(valid.max())) if bounds is None else bounds
        if hi - lo == 0:
            return np.where(np.isnan(arr), np.nan, 0.5).astype(np.float32)   # constant layer (flagged in QC)
        out = (arr - lo) / (hi - lo)
        return (np.clip(out, 0, 1) if bounds is not None else out).astype(np.float32)
    if method == "quantile":
        if bounds is not None:
            raise ValueError("External bounds are only supported with NORM_METHOD = 'minmax'.")
        from scipy.stats import rankdata
        flat = arr.flatten(); nan_mask = np.isnan(flat)
        ranks = np.full_like(flat, np.nan)
        ranks[~nan_mask] = rankdata(flat[~nan_mask]) - 1
        n_valid = (~nan_mask).sum()
        if n_valid > 1:
            ranks[~nan_mask] /= (n_valid - 1)
        return ranks.reshape(arr.shape).astype(np.float32)
    raise ValueError(f"Unknown normalisation method: {method!r}. Use 'minmax' or 'quantile'.")


# ─────────────────────────────────────────────────────────────────────────────
# 2.4  Multifunctionality metrics  (§4.3.3)
#      All functions work on a stack that is NaN in exactly the same pixels for every service
#      (common valid-pixel mask), so the three indices cover the same area.
# ─────────────────────────────────────────────────────────────────────────────
def mf_averaged(norm_stack: np.ndarray) -> np.ndarray:
    """MF_avg = (1/n) Σ ES'_i   (§4.3.3.1) — float array in [0, 1]."""
    return np.mean(norm_stack, axis=0)


def mf_threshold(norm_stack: np.ndarray, threshold: float) -> np.ndarray:
    """
    MF_thr = Σ ES^T_i with ES^T_i = 1 if ES'_i ≥ T else 0   (§4.3.3.2) — counts 0 … n (NaN where any service is missing).
    """
    out = np.sum(norm_stack >= threshold, axis=0).astype(np.float32)
    out[np.any(np.isnan(norm_stack), axis=0)] = np.nan
    return out


def mf_weighted(norm_stack: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """MF_w = Σ w_i · ES'_i with Σ w_i = 1   (§4.3.3.3) — float array in [0, 1]."""
    return np.sum(norm_stack * weights[:, np.newaxis, np.newaxis], axis=0)


# ─────────────────────────────────────────────────────────────────────────────
# 2.5  Figure and GeoTIFF utilities
# ─────────────────────────────────────────────────────────────────────────────
FIGURE_DPI = 150

def save_fig(fig, path: Path) -> None:
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"   Figure saved → {path.name}")


def get_cmap_n(name: str, n: int = None):
    cmap = matplotlib.colormaps[name]
    return cmap.resampled(n) if n else cmap


def imshow_mf(ax, arr, title, cmap="YlGn", vmin=0, vmax=None):
    """Standard imshow call used for all multifunctionality maps (missing pixels in light grey)."""
    cm = get_cmap_n(cmap).copy(); cm.set_bad("#e6e6e6")
    im = ax.imshow(np.ma.masked_invalid(arr), cmap=cm, vmin=vmin,
                   vmax=vmax if vmax is not None else np.nanmax(arr), interpolation="nearest")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axis("off")
    return im


def write_geotiff(arr, path, dtype="float32", nodata=-9999.0, descriptions=None, band_tags=None, tags=None):
    # Write a 2-D array (or a (bands, H, W) stack) as a GeoTIFF on the common grid (profile of REF_RASTER).
    # NaN is written as `nodata` for float outputs; integer outputs must already carry their nodata value.
    # Band descriptions / tags make the files self-documenting.
    arr = np.asarray(arr)
    if arr.ndim == 2:
        arr = arr[None]
    with rasterio.open(REF_RASTER) as ref:
        profile = ref.profile.copy()
    profile.pop("photometric", None)
    profile.update(dtype=dtype, count=arr.shape[0], nodata=nodata, compress="lzw")
    out = (np.where(np.isnan(arr), nodata, arr) if np.issubdtype(arr.dtype, np.floating) else arr).astype(dtype)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out)
        for b in range(arr.shape[0]):
            if descriptions:
                dst.set_band_description(b + 1, descriptions[b])
            if band_tags:
                dst.update_tags(b + 1, **{k: str(v) for k, v in band_tags[b].items()})
        dst.update_tags(**{k: str(v) for k, v in (tags or {}).items()})
    print(f"   GeoTIFF saved → {Path(path).name}")


def to_uint8(arr, nodata=255):
    """NaN-aware conversion of a count / class array to uint8 with an explicit nodata value."""
    return np.where(np.isnan(arr), nodata, arr).astype(np.uint8)


print("Helper functions defined.")

---
## Step 1 — Load and Spatially Harmonise Service Rasters

**D3.5 §4.3.1**

All service rasters are loaded and, if `ALIGN_SERVICES = True`, re-gridded to share a common CRS, resolution,
extent and pixel alignment.  The common grid is that of the **MAES raster** when provided (so that per-type summaries
are exact), otherwise that of the first service raster.  Services given as a list of rasters (e.g. pollinator guilds) are
combined pixel-wise.

A **common valid-pixel mask** is then built: a pixel is used only if it lies inside the ecosystem extent (MAES codes not
excluded) **and** has a value for every service.  All indices of this notebook refer to exactly these pixels.

> *'Prior to index construction, all raster layers are harmonised to ensure: common coordinate reference system (CRS);
> identical spatial resolution; identical spatial extent; consistent pixel alignment.'*  — D3.5 §4.3.1

In [ ]:
ALIGNED_DIR = OUTPUT_DIR / "_aligned_services"
ALIGNED_DIR.mkdir(exist_ok=True)

def spec_paths(spec):
    p = spec["path"]
    return [Path(x) for x in (p if isinstance(p, (list, tuple)) else [p])]

# ── Reference grid: MAES raster if available, else the first existing service raster ──
run_maes = MAES_RASTER is not None and Path(MAES_RASTER).exists()
if run_maes:
    REF_RASTER, ref_name = Path(MAES_RASTER), "MAES raster"
else:
    first = next((p for spec in SERVICES.values() for p in spec_paths(spec) if p.exists()), None)
    if first is None:
        raise FileNotFoundError("No service raster found — check SERVICES.")
    REF_RASTER, ref_name = first, f"first service raster ({first.name})"

with rasterio.open(REF_RASTER) as ref:
    ref_crs, ref_tf, ref_shape = ref.crs, ref.transform, (ref.height, ref.width)
    ref_res = abs(ref.transform.a)
    ref_nodata = ref.nodata
    ref_h, ref_w = ref_shape
pix_ha = get_pixel_area_ha(REF_RASTER)

print(f"Spatial reference : {ref_name}")
print(f"CRS               : {ref_crs}")
print(f"Resolution        : {ref_res:.1f} m")
print(f"Grid size         : {ref_h} × {ref_w} pixels")
print(f"Pixel area        : {pix_ha:.6f} ha\n")

# ── MAES ecosystem extent (optional) ──
maes_arr, code_to_name, excl_mask, unique_codes_maes = None, {}, np.zeros(ref_shape, dtype=bool), np.array([])
if run_maes:
    with rasterio.open(MAES_RASTER) as src:
        maes_arr = src.read(1)
    excl_mask = np.isin(maes_arr, MAES_EXCLUDE_CODES)
    if ref_nodata is not None:
        excl_mask |= (maes_arr == int(ref_nodata))
    unique_codes_maes = np.unique(maes_arr[~excl_mask])
    if MAES_LOOKUP_CSV is not None and Path(MAES_LOOKUP_CSV).exists():
        lk = pd.read_csv(MAES_LOOKUP_CSV)
        code_to_name = dict(zip(lk["maes_code"], lk["ecosystem_type"]))
    print(f"MAES extent       : {int((~excl_mask).sum()):,} pixels, {len(unique_codes_maes)} ecosystem types\n")

# ── Load (and align) every service ──
raw_arrays, service_info = {}, {}
for svc, spec in SERVICES.items():
    paths = spec_paths(spec)
    if not paths or any(not p.exists() for p in paths):
        gone = [str(p) for p in paths if not p.exists()] or ["(no file found)"]
        print(f"  ⚠️  {svc}: NOT FOUND → {', '.join(gone)} — skipped\n")
        continue
    layers, infos = [], []
    for k, p in enumerate(paths):
        arr, info = load_aligned_layer(p, f"{svc}_{k}")
        layers.append(arr); infos.append(info)
    if len(layers) == 1:
        combined = layers[0]
    else:
        stack = np.stack(layers)
        how = spec.get("combine", "mean")
        combined = (np.nanmean(stack, axis=0) if how == "mean"
                    else np.where(np.all(np.isnan(stack), axis=0), np.nan, np.nansum(stack, axis=0))).astype(np.float32)
    raw_arrays[svc] = combined
    service_info[svc] = {"layers": infos, "n_files": len(paths), "same_grid": all(i["same_grid"] for i in infos),
                         "aligned": any(i["aligned"] for i in infos), "res_m": infos[0]["res_m"],
                         "direction": spec["direction"]}

n_svc = len(raw_arrays)
assert n_svc >= 2, "At least two service rasters are required to compute a multifunctionality index."

# ── Common valid-pixel mask ──
mapped_mask = ~excl_mask
valid_mask = mapped_mask.copy()
for svc, arr in raw_arrays.items():
    miss = int((np.isnan(arr) & mapped_mask).sum())
    service_info[svc]["n_missing"] = miss
    valid_mask &= ~np.isnan(arr)
n_mapped_px, n_valid_px = int(mapped_mask.sum()), int(valid_mask.sum())
retained_pct = n_valid_px / n_mapped_px * 100 if n_mapped_px else 0.0

service_arrays = {svc: np.where(valid_mask, arr, np.nan).astype(np.float32) for svc, arr in raw_arrays.items()}
for svc, arr in service_arrays.items():
    valid = arr[~np.isnan(arr)]
    service_info[svc].update({"min": float(valid.min()), "max": float(valid.max()), "mean": float(valid.mean())})
    spec, info = SERVICES[svc], service_info[svc]
    dir_tag = "↑ higher=better" if info["direction"] else "↓ lower=better"
    grid_tag = "re-gridded → common grid" if info["aligned"] else "already on common grid"
    print(f"  ✅ {svc}  ({spec['label']})")
    print(f"     Range : [{info['min']:.4f}, {info['max']:.4f}]  mean = {info['mean']:.4f} {spec['unit']}")
    print(f"     Dir   : {dir_tag}   |  {grid_tag}   |  missing in mapped area: {info['n_missing']:,} px\n")

print(f"{n_svc}/{len(SERVICES)} service rasters loaded.")
print(f"Common valid pixels: {n_valid_px:,} of {n_mapped_px:,} mapped ({retained_pct:.1f} %)")

---
## Step 2 — Directional Harmonisation

**D3.5 §4.3.2.1**

> *'Some outputs represent positive contributions (e.g. carbon storage),
> while others may represent pressures or exports (e.g. nutrient export).
> To ensure comparability, all indicators are transformed such that
> higher values represent higher ecosystem service performance.
> For services representing pressures or disservices, values are inverted
> following normalisation.'*  — D3.5 §4.3.2.1

The inversion is applied **after** normalisation in Step 3 so that
the min–max scaling always sees the original physical values.

In [ ]:
invert_services = [name for name, info in service_info.items() if not info["direction"]]

if invert_services:
    print(f"Services to be inverted after normalisation: {invert_services}")
else:
    print("All services have direction 'higher = better' — no inversion required.")

# A pressure (e.g. nutrient export) that is NOT inverted would count pollution as a benefit
wrong = [n for n in service_info if SERVICES[n]["kind"] == "pressure" and service_info[n]["direction"]]
if wrong:
    print(f"⚠️  {wrong} are pressures but have direction=True — set direction=False in SERVICES.")

---
## Step 3 — Normalisation to [0, 1]

**D3.5 §4.3.2.2**

Each service layer is normalised to a unitless scale [0, 1] using
**min–max scaling** (the primary method specified in D3.5), on the common valid pixels:

$$\text{ES}'_i = \frac{\text{ES}_i - \min(\text{ES}_i)}{\max(\text{ES}_i) - \min(\text{ES}_i)}$$

Services flagged as `direction = False` are then inverted: $\text{ES}'_i \leftarrow 1 - \text{ES}'_i$.

An alternative **quantile-based** normalisation is available via `NORM_METHOD = 'quantile'` and is used in the
sensitivity analysis (Step 5).

> **Comparing years or scenarios.** Min–max bounds computed separately for each run would hide change (every map would
> again span 0–1).  The bounds of this run are therefore saved to `table_normalisation_bounds_<year>.csv`; set
> `NORM_BOUNDS_CSV` to that file in later runs to normalise other years / scenarios on the **same scale**
> (values outside the reference range are clipped and counted).

In [ ]:
ext_bounds = None
if NORM_BOUNDS_CSV is not None and Path(NORM_BOUNDS_CSV).exists():
    if NORM_METHOD != "minmax":
        raise ValueError("NORM_BOUNDS_CSV can only be used with NORM_METHOD = 'minmax'.")
    _b = pd.read_csv(NORM_BOUNDS_CSV).set_index("service")
    ext_bounds = {svc: (float(_b.loc[svc, "min"]), float(_b.loc[svc, "max"])) for svc in service_arrays}
    print(f"Using external normalisation bounds from {Path(NORM_BOUNDS_CSV).name}\n")

norm_arrays, bounds_used, n_clipped = {}, {}, {}
for svc_name, arr in service_arrays.items():
    b = ext_bounds[svc_name] if ext_bounds else None
    if b is not None:
        n_clipped[svc_name] = int(((arr < b[0]) | (arr > b[1])).sum())
    normed = normalise(arr, method=NORM_METHOD, bounds=b)
    if not service_info[svc_name]["direction"]:        # directional inversion (§4.3.2.1)
        normed = 1.0 - normed
    norm_arrays[svc_name] = normed.astype(np.float32)
    bounds_used[svc_name] = b if b is not None else (float(np.nanmin(arr)), float(np.nanmax(arr)))

svc_names  = list(norm_arrays.keys())
norm_stack = np.stack([norm_arrays[n] for n in svc_names], axis=0)     # (n, H, W)

bounds_df = pd.DataFrame({
    "service": svc_names,
    "min": [bounds_used[n][0] for n in svc_names],
    "max": [bounds_used[n][1] for n in svc_names],
    "method": NORM_METHOD,
    "source": "external" if ext_bounds else "computed",
    "direction_higher_is_better": [service_info[n]["direction"] for n in svc_names],
})
out_bounds = OUTPUT_DIR / f"table_normalisation_bounds_{ACCOUNTING_YEAR}.csv"
bounds_df.to_csv(out_bounds, index=False)

print(f"Normalised stack shape: {norm_stack.shape}  (services × rows × cols)   method: {NORM_METHOD}\n")
print(f"  {'Service':<28} {'min':>6}  {'max':>6}  {'mean':>6}  {'inverted':>8}  {'clipped px':>10}")
print("  " + "-" * 72)
for i, name in enumerate(svc_names):
    layer = norm_stack[i]
    print(f"  {name:<28} {np.nanmin(layer):6.3f}  {np.nanmax(layer):6.3f}  {np.nanmean(layer):6.3f}  "
          f"{'yes' if not service_info[name]['direction'] else 'no':>8}  {n_clipped.get(name, 0):>10,}")
print(f"\nBounds saved → {out_bounds.name}  (use it as NORM_BOUNDS_CSV to compare other years/scenarios)")

In [ ]:
# ── Visualise individual normalised service maps ───────────────────────────────
n_cols  = min(3, n_svc)
n_rows  = int(np.ceil(n_svc / n_cols))
cmaps   = ["YlGn", "Blues", "YlOrBr", "RdPu", "Greens", "PuBuGn",
           "Oranges", "BuPu", "GnBu"]

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows), squeeze=False)

for idx, name in enumerate(svc_names):
    ax  = axes[idx // n_cols][idx % n_cols]
    cmp = cmaps[idx % len(cmaps)]
    im  = imshow_mf(ax, norm_stack[idx], name.replace("_", " "),
                    cmap=cmp, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label="norm. value")

# Hide unused axes
for idx in range(n_svc, n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

plt.suptitle(
    f"Normalised Ecosystem Service Layers\n"
    f"{LIVING_SPACE}  |  {ACCOUNTING_YEAR}  |  method: {NORM_METHOD}",
    fontsize=12, fontweight="bold", y=1.01
)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig1_normalised_services_{ACCOUNTING_YEAR}.png")
plt.show()

---
## Step 4 — Compute Multifunctionality Indices

**D3.5 §4.3.3**

Three complementary metrics are computed for every pixel.

### 4a — Averaged Multifunctionality Index (MF_avg)

**D3.5 §4.3.3.1**

$$\text{MF}_{\text{avg}} = \frac{1}{n} \sum_{i=1}^{n} \text{ES}'_i$$

Captures the **mean relative performance** across all ecosystem services.
Values close to 1 indicate consistently high provision; values near 0 indicate
consistently poor provision.

In [ ]:
mf_avg = mf_averaged(norm_stack)

print("MF_avg statistics:")
print(f"  min  : {np.nanmin(mf_avg):.4f}")
print(f"  max  : {np.nanmax(mf_avg):.4f}")
print(f"  mean : {np.nanmean(mf_avg):.4f}")
print(f"  std  : {np.nanstd(mf_avg):.4f}")

### 4b — Threshold-Based Multifunctionality Index (MF_thr)

**D3.5 §4.3.3.2**

$$\text{ES}^T_i = \begin{cases} 1 & \text{if } \text{ES}'_i \geq T \\ 0 & \text{otherwise} \end{cases}
\qquad \text{MF}_{\text{thr}} = \sum_{i=1}^{n} \text{ES}^T_i$$

MF_thr ranges from 0 to $n$ (number of services) and indicates
**how many services are delivered at or above the threshold** $T$.
The primary threshold is $T = $ `THRESHOLD_PRIMARY`.

In [ ]:
mf_thr = mf_threshold(norm_stack, THRESHOLD_PRIMARY)
total_px = int(np.sum(~np.isnan(mf_thr)))

print(f"MF_thr statistics (T = {THRESHOLD_PRIMARY}):")
print(f"  n services : {n_svc}")
print(f"  min        : {np.nanmin(mf_thr):.0f}")
print(f"  max        : {np.nanmax(mf_thr):.0f}")
print(f"  mean       : {np.nanmean(mf_thr):.2f}")
print(f"  std        : {np.nanstd(mf_thr):.2f}")

print("\n  Count   Pixels      % of total")
print("  " + "-" * 36)
for k in range(n_svc + 1):
    n_px = int(np.sum(mf_thr == k))
    print(f"  {k:>5}   {n_px:>8,}   {n_px / total_px * 100:>8.1f}%")

### 4c — Weighted Multifunctionality Index (MF_w)

**D3.5 §4.3.3.3**

$$\text{MF}_w = \sum_{i=1}^{n} w_i \cdot \text{ES}'_i \qquad \text{with} \quad \sum_{i=1}^{n} w_i = 1$$

Weights express stakeholder priorities or policy objectives.
They are read from `SERVICE_WEIGHTS` in Section 1 and automatically normalised.

In [ ]:
# Weight vector aligned with svc_names order (services without weight → 0)
raw_weights = np.array([SERVICES[n].get("weight", 0.0) for n in svc_names], dtype=np.float64)
if raw_weights.sum() == 0:
    raise ValueError("All weights are zero — set at least one non-zero 'weight' in SERVICES.")
norm_weights = raw_weights / raw_weights.sum()

print("Normalised weights (Σ = 1):")
for name, w in zip(svc_names, norm_weights):
    print(f"  {name:<28} : {w:.4f}")

mf_w = mf_weighted(norm_stack, norm_weights)

print("\nMF_w statistics:")
print(f"  min  : {np.nanmin(mf_w):.4f}")
print(f"  max  : {np.nanmax(mf_w):.4f}")
print(f"  mean : {np.nanmean(mf_w):.4f}")
print(f"  std  : {np.nanstd(mf_w):.4f}")

In [ ]:
# ── Figure: side-by-side maps of all three MF indices ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = imshow_mf(axes[0], mf_avg,
                f"MF_avg  (mean norm. performance)\nT = n.a.",
                cmap="YlGn", vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0], fraction=0.04, pad=0.04, label="MF_avg [0–1]")

im1 = imshow_mf(axes[1], mf_thr,
                f"MF_thr  (services ≥ T = {THRESHOLD_PRIMARY})\nrange: 0 – {n_svc}",
                cmap="YlOrRd", vmin=0, vmax=n_svc)
plt.colorbar(im1, ax=axes[1], fraction=0.04, pad=0.04,
             label=f"MF_thr [0–{n_svc}]")

im2 = imshow_mf(axes[2], mf_w,
                "MF_w  (stakeholder-weighted)\nweights from Section 1",
                cmap="PuBuGn", vmin=0, vmax=1)
plt.colorbar(im2, ax=axes[2], fraction=0.04, pad=0.04, label="MF_w [0–1]")

plt.suptitle(
    f"Ecosystem Multifunctionality Indices — {LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig2_multifunctionality_maps_{ACCOUNTING_YEAR}.png")
plt.show()

In [ ]:
# ── Figure: histograms of MF_avg and MF_w; bar chart of MF_thr ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MF_avg histogram
ax = axes[0]
valid_avg = mf_avg[~np.isnan(mf_avg)].flatten()
ax.hist(valid_avg, bins=40, color="#41ab5d", edgecolor="white", linewidth=0.4)
ax.axvline(float(np.nanmean(mf_avg)), color="black", lw=1.5, ls="--",
           label=f"mean = {np.nanmean(mf_avg):.3f}")
ax.set_xlabel("MF_avg", fontsize=10)
ax.set_ylabel("Pixel count", fontsize=10)
ax.set_title("MF_avg distribution", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)

# MF_thr bar chart
ax = axes[1]
counts_thr = [
    int(np.nansum(mf_thr == k)) for k in range(n_svc + 1)
]
colours_thr = get_cmap_n("YlOrRd", n_svc + 1)(
    np.linspace(0, 1, n_svc + 1)
)
bars = ax.bar(range(n_svc + 1), counts_thr,
              color=colours_thr, edgecolor="grey", linewidth=0.4)
ax.set_xlabel(f"Number of services ≥ T={THRESHOLD_PRIMARY}", fontsize=9)
ax.set_ylabel("Pixel count", fontsize=10)
ax.set_title(f"MF_thr distribution  (T = {THRESHOLD_PRIMARY})",
             fontsize=11, fontweight="bold")
ax.set_xticks(range(n_svc + 1))

# MF_w histogram
ax = axes[2]
valid_w = mf_w[~np.isnan(mf_w)].flatten()
ax.hist(valid_w, bins=40, color="#67a9cf", edgecolor="white", linewidth=0.4)
ax.axvline(float(np.nanmean(mf_w)), color="black", lw=1.5, ls="--",
           label=f"mean = {np.nanmean(mf_w):.3f}")
ax.set_xlabel("MF_w", fontsize=10)
ax.set_ylabel("Pixel count", fontsize=10)
ax.set_title("MF_w distribution", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)

plt.suptitle(
    f"Multifunctionality Index Distributions — {LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig3_mf_distributions_{ACCOUNTING_YEAR}.png")
plt.show()

---
## Step 5 — Sensitivity and Uncertainty Analysis

**D3.5 §4.3.4**

To ensure methodological robustness, the following analyses are carried out (all on the common valid pixels):

1. **Threshold sensitivity** — MF_thr is computed across the full `THRESHOLD_RANGE`; the mean landscape-level score is plotted
   as a function of $T$ and the MF_thr map of every threshold is exported (multiband GeoTIFF).
2. **Normalisation method sensitivity** — MF_avg is recomputed with the alternative normalisation (quantile) and the spatial
   correlation with the primary result is reported; the alternative map is exported.
3. **Service redundancy** — Pearson correlation matrix of the normalised layers; high correlations flag potential double counting.
4. **Weight sensitivity** — MF_w is recomputed with equal weights alongside the configured stakeholder weights; the equal-weight map is exported.

Pearson r is reported without p-values: with many spatially autocorrelated pixels significance tests are not informative.
Scenario comparisons (§4.3.4) are done by running the notebook per scenario with the same `NORM_BOUNDS_CSV` (Step 3).

In [ ]:
# ── 5.1  Threshold sensitivity ────────────────────────────────────────────────
thr_maps, thr_means, thr_pct_high = {}, [], []
for T in THRESHOLD_RANGE:
    mf_t = mf_threshold(norm_stack, T)
    thr_maps[T] = mf_t
    thr_means.append(float(np.nanmean(mf_t)))
    thr_pct_high.append(float(np.nansum(mf_t == n_svc) / np.sum(~np.isnan(mf_t)) * 100))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(THRESHOLD_RANGE, thr_means, "o-", color="#e6550d", lw=2)
ax1.axvline(THRESHOLD_PRIMARY, color="grey", ls="--", label=f"Primary T = {THRESHOLD_PRIMARY}")
ax1.set_xlabel("Threshold T", fontsize=10)
ax1.set_ylabel(f"Mean MF_thr (0 – {n_svc})", fontsize=10)
ax1.set_title("Mean MF_thr vs Threshold", fontsize=11, fontweight="bold")
ax1.legend(fontsize=8)

ax2.plot(THRESHOLD_RANGE, thr_pct_high, "s-", color="#31a354", lw=2)
ax2.axvline(THRESHOLD_PRIMARY, color="grey", ls="--", label=f"Primary T = {THRESHOLD_PRIMARY}")
ax2.set_xlabel("Threshold T", fontsize=10)
ax2.set_ylabel("% pixels with ALL services ≥ T", fontsize=10)
ax2.set_title("Full multifunctionality area vs Threshold", fontsize=11, fontweight="bold")
ax2.yaxis.set_major_formatter(PercentFormatter())
ax2.legend(fontsize=8)

plt.suptitle(f"Sensitivity Analysis — Threshold Range\n{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig4_threshold_sensitivity_{ACCOUNTING_YEAR}.png")
plt.show()

thr_df = pd.DataFrame({"threshold": THRESHOLD_RANGE,
                       "mean_mf_thr": [round(v, 4) for v in thr_means],
                       "pct_full_mf": [round(v, 2) for v in thr_pct_high]})
print("\n📋 Threshold sensitivity table:")
display(thr_df)

In [ ]:
# ── 5.2  Normalisation method sensitivity ─────────────────────────────────────
alt_method = "quantile" if NORM_METHOD == "minmax" else "minmax"

alt_norm_stack = np.stack([
    (1.0 - normalise(service_arrays[n], alt_method)) if not service_info[n]["direction"]
    else normalise(service_arrays[n], alt_method)
    for n in svc_names
], axis=0).astype(np.float32)
alt_mf_avg = mf_averaged(alt_norm_stack)

mask_2d = ~np.isnan(mf_avg) & ~np.isnan(alt_mf_avg)
r = float(np.corrcoef(mf_avg[mask_2d], alt_mf_avg[mask_2d])[0, 1])

print("Normalisation sensitivity:")
print(f"  Primary method : {NORM_METHOD}")
print(f"  Alternative    : {alt_method}")
print(f"  Pearson r(MF_avg_primary, MF_avg_alt) = {r:.4f}")
if r >= 0.95:
    print("  ✅ Very high agreement (r ≥ 0.95) — results are robust to normalisation choice.")
elif r >= ROBUST_R_MIN:
    print(f"  ⚠️  Moderate agreement ({ROBUST_R_MIN} ≤ r < 0.95) — consider reporting both methods.")
else:
    print(f"  ⚠️  Low agreement (r < {ROBUST_R_MIN}) — results are sensitive to normalisation choice.")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
imshow_mf(axes[0], mf_avg,     f"MF_avg ({NORM_METHOD})", cmap="YlGn", vmin=0, vmax=1)
imshow_mf(axes[1], alt_mf_avg, f"MF_avg ({alt_method})",  cmap="YlGn", vmin=0, vmax=1)
for ax in axes:
    plt.colorbar(ax.images[0], ax=ax, fraction=0.04, pad=0.04)
plt.suptitle(f"Normalisation Sensitivity  |  Pearson r = {r:.3f}\n{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig5_norm_sensitivity_{ACCOUNTING_YEAR}.png")
plt.show()

In [ ]:
# ── 5.3  Service redundancy — Pearson correlation matrix (common valid pixels) ──
flat = np.stack([norm_stack[i][valid_mask] for i in range(n_svc)], axis=0)
corr_matrix = np.corrcoef(flat) if flat.shape[1] > 1 else np.full((n_svc, n_svc), np.nan)

fig, ax = plt.subplots(figsize=(max(7, n_svc * 1.3), max(6, n_svc * 1.2)))
cax = ax.matshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(cax, ax=ax, fraction=0.04, pad=0.06, label="Pearson r")
short_labels = [n[:16] for n in svc_names]
ax.set_xticks(range(n_svc)); ax.set_yticks(range(n_svc))
ax.set_xticklabels(short_labels, rotation=45, ha="left", fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)
for i in range(n_svc):
    for j in range(n_svc):
        val = corr_matrix[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7,
                    color="black" if abs(val) < 0.7 else "white")
ax.set_title(f"Service Redundancy — Pearson Correlation Matrix\n{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
             fontsize=11, fontweight="bold", pad=20)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig6_service_correlation_{ACCOUNTING_YEAR}.png")
plt.show()

redundant_pairs = [(svc_names[i], svc_names[j], float(corr_matrix[i, j]))
                   for i, j in itertools.combinations(range(n_svc), 2)
                   if not np.isnan(corr_matrix[i, j]) and abs(corr_matrix[i, j]) >= CORR_FLAG]
print(f"Highly correlated service pairs (|r| ≥ {CORR_FLAG}):")
if redundant_pairs:
    for a, b, rr in redundant_pairs:
        print(f"  {a}  ↔  {b} : r = {rr:.3f}")
else:
    print(f"  ✅ No highly correlated pairs detected (|r| < {CORR_FLAG} for all pairs).")

In [ ]:
# ── 5.4  Weight sensitivity — stakeholder vs equal weights ────────────────────
equal_weights = np.ones(n_svc) / n_svc
mf_w_equal    = mf_weighted(norm_stack, equal_weights)

mask_w = ~np.isnan(mf_w) & ~np.isnan(mf_w_equal)
r_w = float(np.corrcoef(mf_w[mask_w], mf_w_equal[mask_w])[0, 1])

print("Weight sensitivity:")
print("  Stakeholder weights : Σ = 1.0  (see SERVICES in Section 1)")
print(f"  Equal weights       : {1/n_svc:.4f} each")
print(f"  Pearson r(MF_w_stake, MF_w_equal) = {r_w:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
imshow_mf(axes[0], mf_w,       "MF_w (stakeholder weights)", cmap="PuBuGn", vmin=0, vmax=1)
imshow_mf(axes[1], mf_w_equal, "MF_w (equal weights)",       cmap="PuBuGn", vmin=0, vmax=1)
for ax in axes:
    plt.colorbar(ax.images[0], ax=ax, fraction=0.04, pad=0.04)
plt.suptitle(f"Weight Sensitivity  |  Pearson r = {r_w:.3f}\n{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig7_weight_sensitivity_{ACCOUNTING_YEAR}.png")
plt.show()

---
## Step 6 — Hotspots, Trade-offs and Zonal Aggregation

**D3.5 §4.3.5**

> *'The multifunctionality index: identifies spatial hotspots of high ecosystem service provision;
> detects areas of trade-offs or underperformance.'*  — D3.5 §4.3.5

- **Hotspots / cold spots** are the top / bottom quartile of MF_avg (`HOTSPOT_QUANTILES`).  They are *relative* classes:
  by construction a fixed share of the pixels falls into each class.
- **Trade-off indicator** `MF_sd` = standard deviation of the normalised services in each pixel (0 = all services at the same
  level; high values = some services high while others are low, i.e. trade-offs).  Together with MF_thr (number of services
  below the threshold = *n* − MF_thr) it shows where multifunctionality is uneven or under-performing.
- **Zonal summaries** are computed per **MAES ecosystem type** and, if `ZONES_VECTOR` is set, per **administrative / basin zone**
  (D3.5 §4.3.1, §4.3.6: statistical summaries at relevant governance levels).

In [ ]:
# ── 6a. Hotspot map and trade-off indicator ───────────────────────────────────
valid_2d = ~np.isnan(mf_avg)
q_lo, q_hi = HOTSPOT_QUANTILES
q25 = float(np.nanpercentile(mf_avg, q_lo))
q75 = float(np.nanpercentile(mf_avg, q_hi))

hotspot_map = np.full(mf_avg.shape, np.nan, dtype=np.float32)
hotspot_map[valid_2d & (mf_avg >= q75)] = 2   # hotspot
hotspot_map[valid_2d & (mf_avg <= q25)] = 0   # cold spot
hotspot_map[valid_2d & (mf_avg > q25) & (mf_avg < q75)] = 1   # intermediate

mf_sd = np.std(norm_stack, axis=0).astype(np.float32)          # trade-off indicator (NaN outside valid pixels)

hotspot_colours = {0: "#d73027", 1: "#ffffbf", 2: "#1a9850"}
hotspot_labels  = {0: f"Cold spot (≤ Q{q_lo})", 1: "Intermediate", 2: f"Hotspot (≥ Q{q_hi})"}

hs_rgb = np.empty((*hotspot_map.shape, 3)); hs_rgb[:] = 0.9
for k, colour in hotspot_colours.items():
    hs_rgb[hotspot_map == k] = mcolors.to_rgb(colour)
hs_patches = [mpatches.Patch(color=hotspot_colours[k], label=hotspot_labels[k]) for k in hotspot_colours]

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
im0 = imshow_mf(axes[0], mf_avg, f"MF_avg — {LIVING_SPACE}  |  {ACCOUNTING_YEAR}", cmap="YlGn", vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0], fraction=0.04, pad=0.04, label="MF_avg")
axes[1].imshow(hs_rgb, interpolation="nearest")
axes[1].set_title(f"Hotspot / Cold-spot Map  (Q{q_lo}={q25:.3f}, Q{q_hi}={q75:.3f})", fontsize=10, fontweight="bold")
axes[1].legend(handles=hs_patches, loc="lower right", fontsize=8, framealpha=0.85)
axes[1].axis("off")
im2 = imshow_mf(axes[2], mf_sd, "Trade-off indicator  (SD of normalised services)", cmap="magma_r", vmin=0)
plt.colorbar(im2, ax=axes[2], fraction=0.04, pad=0.04, label="MF_sd")
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig8_hotspot_map_{ACCOUNTING_YEAR}.png")
plt.show()

print("Hotspot area statistics:")
for k, label in hotspot_labels.items():
    n_px = int(np.nansum(hotspot_map == k))
    print(f"  {label:<30} {n_px:>8,} px   {n_px * pix_ha:>10,.2f} ha   {n_px / total_px * 100:.1f}%")

In [ ]:
# ── 6b. Zonal aggregation: MAES ecosystem types and (optional) administrative zones ──
def zonal_mf_table(zone_arr, codes, name_map, label):
    """One row per zone (valid pixels only): mean indices, hotspot share and mean normalised service values."""
    rows = []
    for code in codes:
        zone = (zone_arr == code) & valid_2d
        n = int(zone.sum())
        if n == 0:
            continue
        rec = {"zone_code": int(code), label: name_map.get(int(code), f"Class {int(code)}"),
               "n_valid_px": n, "area_ha": round(n * pix_ha, 2),
               "mean_mf_avg": round(float(mf_avg[zone].mean()), 4),
               "mean_mf_thr": round(float(mf_thr[zone].mean()), 4),
               "mean_mf_w"  : round(float(mf_w[zone].mean()),   4),
               "mean_mf_sd" : round(float(mf_sd[zone].mean()),  4),
               "pct_hotspot": round(float((hotspot_map[zone] == 2).sum()) / n * 100, 2),
               "pct_all_services_high": round(float((mf_thr[zone] == n_svc).sum()) / n * 100, 2)}
        for i, name in enumerate(svc_names):
            rec[f"norm_{name}"] = round(float(norm_stack[i][zone].mean()), 4)
        rows.append(rec)
    return pd.DataFrame(rows)

zonal_df, zones_df, maes_ids, zone_ids = None, None, None, None

if run_maes:
    maes_ids = maes_arr.astype(np.int64)
    zonal_df = zonal_mf_table(maes_ids, unique_codes_maes, code_to_name, "ecosystem_type")
    zonal_df = zonal_df.rename(columns={"zone_code": "maes_code"})
    print("📋 Zonal Multifunctionality by Ecosystem Type:")
    display(zonal_df[["maes_code", "ecosystem_type", "area_ha", "mean_mf_avg", "mean_mf_thr", "mean_mf_w",
                      "mean_mf_sd", "pct_hotspot"]])
    out_zonal = OUTPUT_DIR / f"table_zonal_multifunctionality_{ACCOUNTING_YEAR}.csv"
    zonal_df.to_csv(out_zonal, index=False)
    print(f"Saved → {out_zonal.name}")
else:
    print("ℹ️  MAES raster not provided or not found — per-ecosystem-type summaries skipped.")

if ZONES_VECTOR is not None and Path(ZONES_VECTOR).exists():
    try:
        import geopandas as gpd
        from rasterio.features import rasterize
        gdf = gpd.read_file(ZONES_VECTOR).to_crs(ref_crs).reset_index(drop=True)
        zone_ids = rasterize([(geom, i + 1) for i, geom in enumerate(gdf.geometry)],
                             out_shape=ref_shape, transform=ref_tf, fill=0, dtype="int32")   # overlaps: last polygon wins
        zone_names = {i + 1: (str(gdf.loc[i, ZONES_NAME_FIELD]) if ZONES_NAME_FIELD in gdf.columns else f"Zone {i + 1}")
                      for i in range(len(gdf))}
        zones_df = zonal_mf_table(zone_ids, sorted(zone_names), zone_names, "zone_name")
        print("\n📋 Zonal Multifunctionality by Zone:")
        display(zones_df[["zone_code", "zone_name", "area_ha", "mean_mf_avg", "mean_mf_thr", "mean_mf_w",
                          "mean_mf_sd", "pct_hotspot"]])
        out_zones = OUTPUT_DIR / f"table_zonal_zones_multifunctionality_{ACCOUNTING_YEAR}.csv"
        zones_df.to_csv(out_zones, index=False)
        print(f"Saved → {out_zones.name}")
    except ImportError:
        print("ℹ️  geopandas is not installed — zone summaries skipped.")
else:
    print("ℹ️  No ZONES_VECTOR provided — administrative-zone summaries skipped.")

In [ ]:
def plot_zonal(df, label_col, fname, title):
    labels = df[label_col].tolist()
    x = np.arange(len(labels)); width = 0.25
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(max(9, len(labels) * 1.5), 10))
    ax_top.bar(x - width, df["mean_mf_avg"], width, label="MF_avg", color="#41ab5d", edgecolor="grey", lw=0.4)
    ax_top.bar(x,         df["mean_mf_w"],   width, label="MF_w",   color="#2171b5", edgecolor="grey", lw=0.4)
    ax_top.bar(x + width, df["mean_mf_thr"] / n_svc, width, label=f"MF_thr / {n_svc}", color="#ef6548",
               edgecolor="grey", lw=0.4)
    ax_top.set_xticks(x); ax_top.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax_top.set_ylabel("Mean index value", fontsize=10); ax_top.set_ylim(0, 1.05)
    ax_top.set_title(f"Mean Multifunctionality Indices by {title}\n{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
                     fontsize=11, fontweight="bold")
    ax_top.legend(fontsize=9)

    norm_cols = [f"norm_{n}" for n in svc_names]
    svc_pal = get_cmap_n("tab10", len(norm_cols)); bar_w = 0.8 / max(len(norm_cols), 1)
    for j, col in enumerate(norm_cols):
        offset = (j - len(norm_cols) / 2 + 0.5) * bar_w
        ax_bot.bar(x + offset, df[col], bar_w * 0.9, label=col.replace("norm_", "")[:22],
                   color=svc_pal(j), edgecolor="grey", lw=0.3)
    ax_bot.set_xticks(x); ax_bot.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax_bot.set_ylabel("Mean normalised ES value (0–1)", fontsize=10); ax_bot.set_ylim(0, 1.1)
    ax_bot.set_title("Mean Normalised Service Values", fontsize=11, fontweight="bold")
    ax_bot.legend(fontsize=7, ncol=3, loc="upper right")
    plt.tight_layout()
    save_fig(fig, OUTPUT_DIR / fname)
    plt.show()

if zonal_df is not None:
    plot_zonal(zonal_df, "ecosystem_type", f"fig9_zonal_multifunctionality_{ACCOUNTING_YEAR}.png", "Ecosystem Type")
if zones_df is not None:
    plot_zonal(zones_df, "zone_name", f"fig10_zones_multifunctionality_{ACCOUNTING_YEAR}.png", "Zone")

---
## Step 7 — Export Deliverables

**D3.5 §4.3.6**

All outputs that are maps are written as **GeoTIFF** on the common grid (CRS, transform and size of the reference raster) in
addition to the PNG figures; charts and tables are exported as PNG / CSV.  Counts and classes are stored as `uint8` (nodata 255),
continuous layers as `float32` (nodata −9999).

| GeoTIFF | Content |
|---|---|
| `mf_avg_<year>.tif`, `mf_w_<year>.tif` | MF_avg, MF_w (0–1) |
| `mf_thr_T<..>_<year>.tif` | MF_thr: number of services ≥ T (0 … n) |
| `mf_tradeoff_sd_<year>.tif` | Trade-off indicator (SD of the normalised services) |
| `hotspot_map_<year>.tif` | 0 = cold spot, 1 = intermediate, 2 = hotspot |
| `norm_<service>_<year>.tif` | Normalised (and direction-harmonised) service layers |
| `mf_thr_sensitivity_<year>.tif` | MF_thr for every threshold of `THRESHOLD_RANGE` (one band each) |
| `mf_sensitivity_scenarios_<year>.tif` | MF_avg with alternative normalisation; MF_w with equal weights |
| `mf_by_ecosystem_type_<year>.tif`, `mf_by_zone_<year>.tif` | Zone means of MF_avg, MF_w, MF_thr, MF_sd painted on every pixel of the zone |

In [ ]:
# ── 7a. GeoTIFF rasters ───────────────────────────────────────────────────────
common_tags = {"living_space": LIVING_SPACE, "accounting_year": ACCOUNTING_YEAR,
               "services": ",".join(svc_names), "n_services": n_svc, "normalisation": NORM_METHOD,
               "source": "LandShift D3.5 section 4.3",
               "nodata_meaning": "outside the ecosystem extent or a service has no value"}
yr = ACCOUNTING_YEAR

write_geotiff(mf_avg, OUTPUT_DIR / f"mf_avg_{yr}.tif", descriptions=["MF_avg"],
              tags={**common_tags, "index": "MF_avg", "formula": "mean of normalised services", "range": "0-1"})
write_geotiff(to_uint8(mf_thr), OUTPUT_DIR / f"mf_thr_T{int(THRESHOLD_PRIMARY*100):03d}_{yr}.tif",
              dtype="uint8", nodata=255, descriptions=["MF_thr"],
              tags={**common_tags, "index": "MF_thr", "threshold_T": THRESHOLD_PRIMARY,
                    "formula": "number of services with normalised value >= T", "range": f"0-{n_svc}"})
write_geotiff(mf_w, OUTPUT_DIR / f"mf_w_{yr}.tif", descriptions=["MF_w"],
              tags={**common_tags, "index": "MF_w", "formula": "sum of weight x normalised service", "range": "0-1",
                    "weights": ",".join(f"{n}={w:.4f}" for n, w in zip(svc_names, norm_weights))})
write_geotiff(mf_sd, OUTPUT_DIR / f"mf_tradeoff_sd_{yr}.tif", descriptions=["MF_sd"],
              tags={**common_tags, "index": "MF_sd", "formula": "standard deviation of normalised services",
                    "meaning": "0 = services equally high/low; high = trade-offs"})
write_geotiff(to_uint8(hotspot_map), OUTPUT_DIR / f"hotspot_map_{yr}.tif", dtype="uint8", nodata=255,
              descriptions=["Hotspot class"],
              tags={**common_tags, "values": f"0=cold spot (<=Q{q_lo}), 1=intermediate, 2=hotspot (>=Q{q_hi}), 255=nodata",
                    "Q_low": round(q25, 4), "Q_high": round(q75, 4)})
for i, name in enumerate(svc_names):
    write_geotiff(norm_stack[i], OUTPUT_DIR / f"norm_{name}_{yr}.tif", descriptions=[SERVICES[name]["label"]],
                  tags={**common_tags, "service": name, "unit_original": SERVICES[name]["unit"],
                        "inverted": not service_info[name]["direction"],
                        "bounds_min": round(bounds_used[name][0], 6), "bounds_max": round(bounds_used[name][1], 6)})

# Sensitivity outputs
write_geotiff(np.stack([to_uint8(thr_maps[T]) for T in THRESHOLD_RANGE]),
              OUTPUT_DIR / f"mf_thr_sensitivity_{yr}.tif", dtype="uint8", nodata=255,
              descriptions=[f"MF_thr T={T}" for T in THRESHOLD_RANGE],
              band_tags=[{"threshold_T": T} for T in THRESHOLD_RANGE], tags={**common_tags, "index": "MF_thr"})
write_geotiff(np.stack([alt_mf_avg, mf_w_equal]), OUTPUT_DIR / f"mf_sensitivity_scenarios_{yr}.tif",
              descriptions=[f"MF_avg ({alt_method} normalisation)", "MF_w (equal weights)"],
              band_tags=[{"scenario": f"{alt_method} normalisation", "pearson_r_vs_primary": round(r, 4)},
                         {"scenario": "equal weights", "pearson_r_vs_primary": round(r_w, 4)}],
              tags=common_tags)

# Zone-level maps (type-level / zone-level means painted on every pixel of the zone)
def paint_zone_means(zone_arr, df, code_col):
    bands = []
    for col in ("mean_mf_avg", "mean_mf_w", "mean_mf_thr", "mean_mf_sd"):
        band = np.full(zone_arr.shape, np.nan, dtype=np.float32)
        for code, val in zip(df[code_col], df[col]):
            band[(zone_arr == code) & valid_2d] = val
        bands.append(band)
    return np.stack(bands)

zone_band_desc = ["MF_avg (zone mean)", "MF_w (zone mean)", "MF_thr (zone mean)", "MF_sd (zone mean)"]
if zonal_df is not None:
    write_geotiff(paint_zone_means(maes_ids, zonal_df, "maes_code"), OUTPUT_DIR / f"mf_by_ecosystem_type_{yr}.tif",
                  descriptions=zone_band_desc, tags={**common_tags, "product": "Mean multifunctionality by MAES ecosystem type",
                                                     "note": "zone means painted on every valid pixel of the zone"})
if zones_df is not None:
    write_geotiff(paint_zone_means(zone_ids, zones_df, "zone_code"), OUTPUT_DIR / f"mf_by_zone_{yr}.tif",
                  descriptions=zone_band_desc, tags={**common_tags, "product": "Mean multifunctionality by administrative zone",
                                                     "note": "zone means painted on every valid pixel of the zone"})

In [ ]:
# ── 7b. Summary statistics table ─────────────────────────────────────────────
summary_rows = []
for label, arr in [("MF_avg", mf_avg), ("MF_thr", mf_thr), ("MF_w", mf_w), ("MF_sd", mf_sd)]:
    valid = arr[~np.isnan(arr)]
    summary_rows.append({
        "index"  : label, "n_valid": int(valid.size),
        "min"    : round(float(valid.min()), 4), "q25": round(float(np.percentile(valid, 25)), 4),
        "median" : round(float(np.median(valid)), 4), "mean": round(float(valid.mean()), 4),
        "q75"    : round(float(np.percentile(valid, 75)), 4), "max": round(float(valid.max()), 4),
        "std"    : round(float(valid.std()), 4),
    })
summary_df = pd.DataFrame(summary_rows)
print("📋 Multifunctionality Index Summary Statistics:")
display(summary_df)
out_stats = OUTPUT_DIR / f"table_mf_statistics_{ACCOUNTING_YEAR}.csv"
summary_df.to_csv(out_stats, index=False)
print(f"Saved → {out_stats.name}")

In [ ]:
# ── 7c. Threshold sensitivity table ──────────────────────────────────────────
out_thr = OUTPUT_DIR / f"table_threshold_sensitivity_{ACCOUNTING_YEAR}.csv"
thr_df.to_csv(out_thr, index=False)
print(f"Saved → {out_thr.name}")

# ── 7d. Correlation matrix ────────────────────────────────────────────────────
out_corr = OUTPUT_DIR / f"table_service_correlation_{ACCOUNTING_YEAR}.csv"
pd.DataFrame(corr_matrix.round(4), index=svc_names, columns=svc_names).to_csv(out_corr)
print(f"Saved → {out_corr.name}")

# ── 7e. Weight / service table ────────────────────────────────────────────────
weight_df = pd.DataFrame({
    "service"      : svc_names,
    "label"        : [SERVICES[n]["label"] for n in svc_names],
    "unit"         : [SERVICES[n]["unit"] for n in svc_names],
    "kind"         : [SERVICES[n]["kind"] for n in svc_names],
    "group"        : [SERVICES[n]["group"] for n in svc_names],
    "direction_higher_is_better": [SERVICES[n]["direction"] for n in svc_names],
    "raw_weight"   : raw_weights.round(4),
    "norm_weight"  : norm_weights.round(6),
    "equal_weight" : equal_weights.round(6),
})
out_weights = OUTPUT_DIR / f"table_weights_{ACCOUNTING_YEAR}.csv"
weight_df.to_csv(out_weights, index=False)
print(f"Saved → {out_weights.name}")

---
## Section 8 — Quality Checks

**D3.5 §4.3.1 / §4.3.4** — *Spatial harmonisation, sensitivity and uncertainty analysis*

Nine automated checks are run before the notebook finalises:

1. Spatial grid consistency of the inputs
2. Services loaded / valid-pixel coverage (common mask)
3. Normalised value range [0, 1] and constant layers
4. Weights sum to 1
5. Direction vs kind (pressures must be inverted) and double-counting groups
6. Service redundancy (correlation)
7. Robustness to normalisation method and weights
8. All exported GeoTIFFs share the reference grid
9. Accounting year in every output filename

In [ ]:
print("═" * 65)
print("TECHNICAL QUALITY CHECKS  (D3.5 §4.3)")
print("═" * 65)
issues = []

# 1. Spatial grid consistency
print(f"\n1. Spatial grid consistency  (reference: {ref_name}, {ref_res:.1f} m)")
for name, info in service_info.items():
    for lay in info["layers"]:
        if lay["same_grid"]:
            tag = "✅"
        elif lay["aligned"]:
            tag = "↻ re-gridded to the reference grid"
        else:
            tag = "⚠️  MISMATCH"; issues.append(f"Grid mismatch: {name} / {lay['file']}")
        print(f"   {name:<28} {lay['file']:<34} {lay['res_m']:7.1f} m  {tag}")

# 2. Services loaded and coverage
print("\n2. Services loaded and valid-pixel coverage (common mask)")
missing_svc = [s for s in SERVICES if s not in service_arrays]
if missing_svc:
    print(f"   ⚠️  Services not loaded: {missing_svc}")
    issues.append(f"Services not loaded: {missing_svc} (weights are re-normalised over the loaded services).")
for name, info in service_info.items():
    print(f"   {name:<28} missing in mapped area: {info['n_missing']:>8,} px")
tag = "✅" if retained_pct >= MIN_RETAINED_PCT else "⚠️  LOW"
print(f"   Common valid pixels: {retained_pct:.1f} % of the mapped area  {tag}")
if retained_pct < MIN_RETAINED_PCT:
    issues.append(f"Only {retained_pct:.1f} % of the mapped area has values for ALL services (< {MIN_RETAINED_PCT:.0f} %).")

# 3. Normalised range and constant layers
print("\n3. Normalised value range [0, 1] and constant layers")
for i, name in enumerate(svc_names):
    lo, hi = float(np.nanmin(norm_stack[i])), float(np.nanmax(norm_stack[i]))
    ok = lo >= -1e-6 and hi <= 1 + 1e-6
    const = service_info[name]["max"] == service_info[name]["min"]
    tag = "✅" if ok and not const else ("⚠️  CONSTANT LAYER" if const else "⚠️  OUT OF RANGE")
    print(f"   {name:<28} [{lo:.4f}, {hi:.4f}]  {tag}")
    if not ok:
        issues.append(f"Normalised values outside [0,1] for {name}.")
    if const:
        issues.append(f"{name} is constant — it carries no information (normalised to 0.5).")

# 4. Weights
print("\n4. Weight normalisation (Σ = 1)")
w_sum = norm_weights.sum()
tag = "✅" if abs(w_sum - 1.0) < 1e-9 else "⚠️  DOES NOT SUM TO 1"
print(f"   Σ norm_weights = {w_sum:.10f}  {tag}")
if abs(w_sum - 1.0) >= 1e-9:
    issues.append("Normalised weights do not sum to 1.")

# 5. Direction vs kind, double counting
print("\n5. Direction vs kind, double-counting groups")
dir_ok = True
for n in svc_names:
    if SERVICES[n]["kind"] == "pressure" and SERVICES[n]["direction"]:
        dir_ok = False
        print(f"   ⚠️  {n}: a pressure with direction=True — pollution would count as a benefit")
        issues.append(f"{n} is a pressure but direction=True (should be False).")
    if SERVICES[n]["kind"] != "pressure" and not SERVICES[n]["direction"]:
        print(f"   ℹ️  {n}: direction=False although kind='{SERVICES[n]['kind']}' — check.")
groups = {}
for n in svc_names:
    groups.setdefault(SERVICES[n]["group"], []).append(n)
for g, s in groups.items():
    if len(s) > 1:
        dir_ok = False
        print(f"   ⚠️  Group '{g}': {s} overlap (double counting)")
        issues.append(f"Overlapping services in group '{g}': {s}.")
if dir_ok:
    print("   ✅ Directions consistent with service kinds; no overlapping service groups.")

# 6. Redundancy
print(f"\n6. Service redundancy (|r| ≥ {CORR_FLAG})")
if redundant_pairs:
    for a, b, rr in redundant_pairs:
        print(f"   ⚠️  {a} ↔ {b}: r = {rr:.3f}")
        issues.append(f"Redundant services {a} ↔ {b} (r = {rr:.2f}).")
else:
    print("   ✅ No redundant service pairs.")

# 7. Robustness
print(f"\n7. Robustness of MF (Pearson r ≥ {ROBUST_R_MIN})")
for label, val in [(f"MF_avg {NORM_METHOD} vs {alt_method}", r), ("MF_w stakeholder vs equal weights", r_w)]:
    ok = val >= ROBUST_R_MIN
    print(f"   {label:<40} r = {val:.3f}  {'✅' if ok else '⚠️  SENSITIVE'}")
    if not ok:
        issues.append(f"Result sensitive to the choice: {label} (r = {val:.2f}).")

# 8. Exported GeoTIFFs on the reference grid
print("\n8. Exported GeoTIFFs share the reference grid")
bad_tifs = []
tifs = sorted(OUTPUT_DIR.glob("*.tif"))
for f in tifs:
    with rasterio.open(f) as d:
        if not (str(d.crs) == str(ref_crs) and (d.height, d.width) == ref_shape
                and np.allclose(tuple(d.transform)[:6], tuple(ref_tf)[:6], atol=1e-6)):
            bad_tifs.append(f.name)
if bad_tifs:
    print(f"   ⚠️  Different grid: {bad_tifs}"); issues.append(f"GeoTIFFs not on the reference grid: {bad_tifs}")
else:
    print(f"   ✅ All {len(tifs)} GeoTIFFs match the reference CRS, extent and resolution.")

# 9. Year stamp
print(f"\n9. Accounting year ({ACCOUNTING_YEAR}) embedded in all output filenames")
no_year = [f.name for f in list(OUTPUT_DIR.glob("*.tif")) + list(OUTPUT_DIR.glob("table*.csv")) if str(ACCOUNTING_YEAR) not in f.name]
if no_year:
    print(f"   ⚠️  Missing year stamp: {no_year}"); issues.append("Some output files lack the accounting year in their name.")
else:
    print("   ✅ All GeoTIFF / CSV outputs are year-stamped.")

print("\n" + "═" * 65)
if issues:
    print(f"⚠️  {len(issues)} issue(s) require attention:")
    for i, iss in enumerate(issues, 1):
        print(f"   {i}. {iss}")
else:
    print("✅ All checks passed — multifunctionality outputs are ready.")
print("═" * 65)

---
## Section 9 — Grand-Total Summary

In [ ]:
print("GRAND SUMMARY — Ecosystem Multifunctionality")
print("-" * 65)
print(f"Living Space       : {LIVING_SPACE}")
print(f"Accounting year    : {ACCOUNTING_YEAR}")
print(f"Services included  : {n_svc}  {svc_names}")
print(f"Normalisation      : {NORM_METHOD}" + ("  (external bounds)" if ext_bounds else ""))
print(f"Primary threshold  : {THRESHOLD_PRIMARY}")
print(f"Grid size          : {ref_h} × {ref_w}  ({ref_h * ref_w:,} pixels)")
print(f"Pixel area         : {pix_ha:.6f} ha")
total_mapped = n_valid_px * pix_ha
print(f"Valid area         : {total_mapped:,.2f} ha  ({total_mapped/100:,.2f} km²)  = {retained_pct:.1f} % of the mapped area")
print()
print(f"  {'Index':<12}  {'min':>7}  {'mean':>7}  {'max':>7}  {'std':>7}")
print("  " + "-" * 45)
for row in summary_df.itertuples():
    print(f"  {row.index:<12}  {row.min:>7.4f}  {row.mean:>7.4f}  {row.max:>7.4f}  {row.std:>7.4f}")
print()
hs_pct = float(np.nansum(hotspot_map == 2) / total_px * 100)
print(f"  Hotspot area (≥ Q{q_hi}) : {hs_pct:.1f}% of the valid area")
print(f"  Normalisation sensitivity (r) : {r:.3f}")
print(f"  Weight sensitivity (r)        : {r_w:.3f}")
print()
print(f"All outputs in: {OUTPUT_DIR.resolve()}")

---
## Section 10 — Processing Report

In [ ]:
ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

lines = [
    "# LandShift D3.5 — Section 4.3 Ecosystem Multifunctionality Processing Report", "",
    "## Run metadata", "",
    "| Key | Value |", "|---|---|",
    f"| Generated | {ts} |",
    f"| Living Space | {LIVING_SPACE} |",
    f"| Accounting year | {ACCOUNTING_YEAR} |",
    f"| Demo mode | {DEMO_MODE} |",
    f"| Services | {n_svc} |",
    f"| Normalisation | {NORM_METHOD}" + (" (external bounds)" if ext_bounds else "") + " |",
    f"| Primary threshold | {THRESHOLD_PRIMARY} |",
    f"| Reference grid | {ref_name}: `{REF_RASTER}` ({ref_res:.1f} m, {ref_crs}) |",
    f"| Common valid pixels | {n_valid_px:,} of {n_mapped_px:,} mapped ({retained_pct:.1f} %) |",
    "", "## Input services", "",
    "| Service | Label | Kind | Group | Direction | Unit | Re-gridded |", "|---|---|---|---|---|---|---|",
]
for name in svc_names:
    s = SERVICES[name]
    lines.append(f"| {name} | {s['label']} | {s['kind']} | {s['group']} | "
                 f"{'↑ higher=better' if s['direction'] else '↓ lower=better (inverted)'} | {s['unit']} | {service_info[name]['aligned']} |")

lines += ["", "## Normalisation bounds", "", "| Service | min | max | source |", "|---|---|---|---|"]
for _, b in bounds_df.iterrows():
    lines.append(f"| {b['service']} | {b['min']:.6g} | {b['max']:.6g} | {b['source']} |")

lines += ["", "## Normalised weights (MF_w)", "", "| Service | Raw weight | Normalised weight |", "|---|---|---|"]
for name, rw, nw in zip(svc_names, raw_weights, norm_weights):
    lines.append(f"| {name} | {rw:.4f} | {nw:.6f} |")

lines += ["", "## Summary statistics", "", "| Index | min | mean | max | std |", "|---|---|---|---|---|"]
for row in summary_df.itertuples():
    lines.append(f"| {row.index} | {row.min} | {row.mean} | {row.max} | {row.std} |")

lines += ["", "## Sensitivity analysis", "",
          f"- Normalisation sensitivity (r, {NORM_METHOD} vs {alt_method}): **{r:.4f}**",
          f"- Weight sensitivity (r, stakeholder vs equal): **{r_w:.4f}**",
          f"- Threshold range tested: {THRESHOLD_RANGE}",
          f"- Redundant service pairs (|r| ≥ {CORR_FLAG}): {[(a, b, round(c, 2)) for a, b, c in redundant_pairs] or 'none'}",
          "", "## Quality check summary", ""]
lines += [f"- ⚠️ {iss}" for iss in issues] if issues else ["- ✅ All checks passed."]

lines += [
    "", "## Output files", "", f"Directory: `{OUTPUT_DIR.resolve()}/`", "",
    "| File | Description |", "|---|---|",
    f"| `mf_avg_{ACCOUNTING_YEAR}.tif` | Averaged Multifunctionality Index (GeoTIFF, 0–1) |",
    f"| `mf_thr_T{int(THRESHOLD_PRIMARY*100):03d}_{ACCOUNTING_YEAR}.tif` | Threshold-Based MF: number of services ≥ T (GeoTIFF, uint8) |",
    f"| `mf_w_{ACCOUNTING_YEAR}.tif` | Weighted MF (GeoTIFF, 0–1) |",
    f"| `mf_tradeoff_sd_{ACCOUNTING_YEAR}.tif` | Trade-off indicator, SD of normalised services (GeoTIFF) |",
    f"| `hotspot_map_{ACCOUNTING_YEAR}.tif` | Hotspot / cold-spot classes (GeoTIFF, uint8) |",
    f"| `norm_<service>_{ACCOUNTING_YEAR}.tif` | Normalised, direction-harmonised service layers (GeoTIFF) |",
    f"| `mf_thr_sensitivity_{ACCOUNTING_YEAR}.tif` | MF_thr for each threshold, one band per T (GeoTIFF) |",
    f"| `mf_sensitivity_scenarios_{ACCOUNTING_YEAR}.tif` | MF_avg with alternative normalisation; MF_w with equal weights (GeoTIFF) |",
    f"| `mf_by_ecosystem_type_{ACCOUNTING_YEAR}.tif` | Zone means by MAES type, 4 bands (GeoTIFF, if MAES provided) |",
    f"| `mf_by_zone_{ACCOUNTING_YEAR}.tif` | Zone means by administrative zone, 4 bands (GeoTIFF, if zones provided) |",
    f"| `table_normalisation_bounds_{ACCOUNTING_YEAR}.csv` | Min–max bounds per service (reuse as `NORM_BOUNDS_CSV`) |",
    f"| `table_mf_statistics_{ACCOUNTING_YEAR}.csv` | Summary statistics for all MF indices |",
    f"| `table_zonal_multifunctionality_{ACCOUNTING_YEAR}.csv` | Zonal MF by ecosystem type |",
    f"| `table_zonal_zones_multifunctionality_{ACCOUNTING_YEAR}.csv` | Zonal MF by administrative zone |",
    f"| `table_threshold_sensitivity_{ACCOUNTING_YEAR}.csv` | MF_thr across threshold range |",
    f"| `table_service_correlation_{ACCOUNTING_YEAR}.csv` | Pearson correlation matrix |",
    f"| `table_weights_{ACCOUNTING_YEAR}.csv` | Service metadata, directions and weights |",
    "| `fig1_*.png` … `fig10_*.png` | Normalised maps, MF maps, distributions, sensitivity figures, hotspots / trade-offs, zonal charts |",
    "", "---",
    "*LandShift — Horizon Europe Grant Agreement No. 101182007*  ",
    "*D3.5 Ecosystem services & landscape ecology — Section 4.3*  ",
    "*University of Geneva (UNIGE)*",
]
rpt = OUTPUT_DIR / f"processing_report_{ACCOUNTING_YEAR}.md"
rpt.write_text("\n".join(lines), encoding="utf-8")
print(f"Processing report saved → {rpt.name}")
print(f"\n✅  Section 4.3 notebook complete.")
print(f"    All outputs in: {OUTPUT_DIR.resolve()}")

---
## FAIR Compliance Summary

| Principle | Implementation |
|-----------|----------------|
| **Findable** | Year-stamped filenames; Markdown report indexes every output file |
| **Accessible** | Open formats only: GeoTIFF rasters (self-documenting band descriptions and tags), CSV tables, PNG figures, MD report |
| **Interoperable** | MAES (EEA standard) for zonal aggregation; SEEA-EA-compatible physical units throughout; normalisation equations from published literature |
| **Reusable** | Section 1 isolates all user parameters; `DEMO_MODE` enables testing without real data; normalisation bounds are exported so other years / scenarios can be compared on the same scale; three metrics + sensitivity analyses support transparent, reproducible reporting |

All Tier 2 and Tier 3 notebooks will be published in the **LandShift Zenodo repository**
at M30 (April 2027) with a DOI, as required by Horizon Europe.

### Key references (D3.5 §4.3)

- Byrnes et al. (2014) — multifunctionality metrics framework. *Nature Ecology & Evolution* doi:10.1038/s41559-017-0461-7
- Zavaleta et al. (2010) — threshold-based multifunctionality. *Science* doi:10.1126/science.1215442
- Manning et al. (2018) — weighted multifunctionality. *Journal of Environmental Management* doi:10.1016/j.jenvman.2020.111068

---
*LandShift — Horizon Europe Grant Agreement No. 101182007*  
*D3.5 Ecosystem services & landscape ecology — Section 4.3*  
*University of Geneva (UNIGE)*